<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
# <font color='blue'>Data Science Academy</font>
## <font color='blue'>IA Generativa e LLMs Para Processamento de Linguagem Natural</font>
## <font color='blue'>Estudo de Caso 2</font>
## <font color='blue'>Criando Aplicações Inteligentes com LangChain e LLMs</font>

## Instalando e Carregando Pacotes

In [10]:
# Para atualizar um pacote, execute o comando abaixo no terminal ou prompt de comando:
# pip install -U nome_pacote

# Para instalar a versão exata de um pacote, execute o comando abaixo no terminal ou prompt de comando:
# !pip install nome_pacote==versão_desejada

# Depois de instalar ou atualizar o pacote, reinicie o jupyter notebook.

# Instala o pacote watermark.
# Esse pacote é usado para gravar as versões de outros pacotes usados neste jupyter notebook.
!pip install -q -U watermark

O procedimento de instalação dos pacotes e dependências está no arquivo LEIAME.txt

In [11]:
# Langchain
!pip install -q langchain

In [12]:
# Langchain para conectar no LLM do Google Gemini
!pip install -q langchain-google-genai

In [13]:
# VectorDB
!pip install -q chromadb

In [14]:
# Dependência para o Langchain
!pip install -q sqlalchemy

In [15]:
# Define o USER_AGENT
import os
os.environ["USER_AGENT"] = "Estudo de Caso 2"

In [16]:
# Imports
import os
import textwrap
import chromadb
import langchain
import sqlalchemy
import langchain_google_genai
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.chains import SequentialChain
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.document_loaders import WebBaseLoader
from langchain.indexes import VectorstoreIndexCreator

from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA

import warnings
warnings.filterwarnings('ignore')

In [17]:
# Versões dos pacotes usados neste jupyter notebook
%reload_ext watermark
%watermark -a "Data Science Academy"

Author: Data Science Academy



## Conhecendo o LangChain

Veja o e-book no Capítulo 12. Visite também a documentação oficial:

https://python.langchain.com/docs/get_started/introduction

## Definindo o LLM de Trabalho

In [ ]:
# Coloque aqui sua API do Google Gemini
os.environ['GOOGLE_API_KEY'] = ""

In [19]:
# Define o LLM
# Cria uma instância de um Large Language Model (LLM), especificamente um fornecido pelo Google Gemini
dsa_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)

A temperatura é um hiperparâmetro que influencia a aleatoriedade das respostas geradas pelo modelo. Um valor mais alto de temperatura (geralmente variando entre 0 e 1) promove respostas mais criativas e variadas. Por outro lado, uma temperatura baixa tende a fazer com que o modelo produza respostas mais determinísticas e possivelmente mais previsíveis.

In [20]:
# Envia o prompt para o LLM e captura a resposta
nome = dsa_llm.invoke("Quero abrir um restaurante de comida japonesa. Sugira um nome chique para isso.")

Neste contexto, a string "Quero abrir um restaurante de comida japonesa. Sugira um nome chique para isso." serve como a prompt ou entrada para o modelo de linguagem. Ela descreve a tarefa que o usuário deseja que o modelo execute: a geração criativa de um nome para um novo restaurante de comida japonesa. O modelo utilizará seu treinamento em linguagem natural e conhecimento prévio para gerar uma resposta que atenda a essa solicitação.

In [21]:
print(nome)

content='Que ótima iniciativa! Um nome chique para um restaurante japonês deve evocar elegância, tradição, natureza, e um toque de modernidade ou exclusividade.\n\nAqui estão algumas sugestões, com as respectivas traduções/significados para te ajudar a escolher a que melhor se alinha com a sua visão:\n\n**Nomes que evocam Natureza e Elementos:**\n\n1.  **Mizuya (水屋):** "Casa da Água" ou "Sala da Água". Evoca pureza, fluidez e a importância da água na culinária e estética japonesa. Muito elegante.\n2.  **Tsuki (月):** "Lua". Simples, poético e evoca serenidade e beleza noturna.\n3.  **Sora (空):** "Céu". Transmite amplitude, liberdade e leveza.\n4.  **Nami Kai (波海):** "Ondas do Mar/Oceano". Sugere movimento, profundidade e a riqueza dos frutos do mar.\n5.  **Aki (秋):** "Outono". Evoca maturidade, colheita, cores quentes e uma certa sofisticação.\n6.  **Komorebi (木漏れ日):** "Luz do sol filtrada pelas árvores". É mais longo, mas incrivelmente poético e evoca uma atmosfera serena e acolhedora.

## Usando Prompt Templates

Os modelos de prompt (Prompt Templates) no contexto do LangChain referem-se a formas estruturadas de formatação da entrada para grandes modelos de linguagem (LLMs) para melhorar seu desempenho e adesão aos comportamentos desejados.

Um modelo de prompt define uma sequência de modelo com variáveis de espaço reservado que podem ser preenchidas dinamicamente. Isso permite que você construa prompts de maneira consistente e programática, em vez de codificar prompts completos.

Os modelos de prompt no LangChain fornecem uma maneira estruturada e extensível de interface com LLMs, facilitando a exploração e otimização de estratégias de prompt para melhorar o desempenho do modelo de linguagem em tarefas ou domínios específicos.

In [22]:
# Define o prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables = ['culinaria'],
    template = "Quero abrir um restaurante de comida {culinaria}. Sugira um nome chique para isso.")

A linha de código acima define um PromptTemplate, uma estrutura que permite a criação de prompts dinâmicos para serem utilizados com Large Language Models (LLMs). Essa abordagem é particularmente útil quando se deseja gerar prompts personalizados baseados em variáveis específicas ou quando se pretende reutilizar um formato de prompt com diferentes conjuntos de dados. 

**input_variables = ['culinaria']**: Define uma lista de variáveis que podem ser usadas para preencher o template. Neste caso, há uma única variável chamada 'culinaria'. Essa variável atua como um placeholder que será substituído por um valor específico quando o template for utilizado.

In [23]:
# Utiliza o template definido anteriormente para gerar um prompt específico, 
# inserindo o valor "Italiana" no lugar da variável culinaria
p = dsa_prompt_template_name.format(culinaria = "Italiana")

In [24]:
print(p)

Quero abrir um restaurante de comida Italiana. Sugira um nome chique para isso.


## Sequências de Operações com LLMChain

As chains no LangChain são sequências de operações que podem processar entradas e gerar saídas combinando vários componentes, incluindo grandes modelos de linguagem (LLMs), outras cadeias e ferramentas ou utilitários especializados.

Um LLMChain é um tipo de cadeia que permite interagir com um grande modelo de linguagem (LLM) de forma estruturada. Ele fornece uma interface simples para passar entradas para o LLM e recuperar suas saídas.

O LLMChain serve como um bloco de construção para muitas outras construções no LangChain, como agentes, ferramentas e tipos de cadeia mais avançados. Ao encapsular a lógica de interação do LLM em um componente reutilizável e extensível, o LLMChain simplifica o processo de construção de aplicativos que aproveitam grandes modelos de linguagem.

In [25]:
# Cria a chain
chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

A linha de código acima cria uma instância de LLMChain, uma classe destinada a encadear ou sequenciar operações utilizando um LLM. Essa instância é configurada para usar um modelo de linguagem específico e um template de prompt predefinido. 

In [26]:
# Invoca a chain passando um parâmetro para o prompt
chain.invoke("Mexicana")

{'culinaria': 'Mexicana',
 'text': 'Que ótima ideia! Para um restaurante de comida mexicana chique, o nome deve evocar sofisticação, tradição com um toque moderno, ou uma sensação de descoberta e exclusividade.\n\nAqui estão algumas sugestões, divididas por estilos, para inspirar:\n\n---\n\n**1. Nomes que Evocam Elegância e Tradição Refinada:**\n\n*   **Alma y Sabor** (Alma e Sabor): Evoca a essência e o coração da culinária.\n*   **Mesa de Oro** (Mesa de Ouro): Sugere luxo, preciosidade e uma experiência gastronômica de alto nível.\n*   **El Jardín Secreto** (O Jardim Secreto): Implica um local exclusivo, íntimo e de descobertas culinárias.\n*   **Canto del Fuego** (Canto do Fogo): Poético, remete ao preparo artesanal e paixão.\n*   **Piedra y Plata** (Pedra e Prata): Combina elementos naturais e preciosos, remetendo à arte e elegância mexicanas.\n*   **La Sal y el Cielo** (O Sal e o Céu): Evoca a pureza dos ingredientes e a grandiosidade da experiência.\n*   **Esencia Mexicana**: Sim

In [27]:
# Cria a chain e ativa o verbose
chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, verbose = True)

In [28]:
# Invoca a chain passando um parâmetro para o prompt
chain.invoke("Mexicana")



> Entering new LLMChain chain...
Prompt after formatting:
Quero abrir um restaurante de comida Mexicana. Sugira um nome chique para isso.

> Finished chain.


{'culinaria': 'Mexicana',
 'text': 'Que ótima ideia! Para um restaurante de comida mexicana chique, queremos nomes que evoquem sofisticação, autenticidade (mas elevada), um toque de mistério ou elegância, e que sejam fáceis de lembrar.\n\nAqui estão algumas sugestões, divididas por estilos, para você escolher:\n\n---\n\n### Nomes Elegantes e Clássicos (em Espanhol)\n\n1.  **El Refugio** (O Refúgio) – Sugere um escape sofisticado.\n2.  **La Mesa Escondida** (A Mesa Escondida) – Evoca exclusividade e descoberta.\n3.  **Oro y Maíz** (Ouro e Milho) – Combina luxo com o ingrediente essencial.\n4.  **Sabor Secreto** (Sabor Secreto) – Intriga e promete uma experiência única.\n5.  **Alma y Sazón** (Alma e Tempero) – Reflete paixão e autenticidade refinada.\n6.  **Cielo y Tierra** (Céu e Terra) – Grandioso, conecta com a natureza e ingredientes frescos.\n7.  **La Pérgola** – Remete a um espaço elegante e aberto, geralmente com jardim.\n8.  **El Jardín Secreto** – Evoca beleza, exclusividade e u

## Simple Sequential Chain

Um SimpleSequentialChain no LangChain é um tipo de cadeia que executa uma sequência de componentes (por exemplo, LLMs, ferramentas, outras cadeias) em uma ordem predefinida. É um dos tipos de cadeia mais básicos e comumente usados no LangChain.

Um exemplo de caso de uso de SimpleSequentialChain poderia ser um sistema de resposta a perguntas onde:

- O primeiro componente é um LLM que analisa a questão de entrada.
- O segundo componente é uma ferramenta que recupera documentos relevantes de um banco de dados.
- O terceiro componente é outro LLM que gera uma resposta com base na pergunta e nos documentos recuperados.

Ao encadear esses componentes em um SimpleSequentialChain, você pode criar um sistema mais complexo e capaz, mantendo uma arquitetura modular e extensível.

Embora SimpleSequentialChain seja útil para fluxos de trabalho lineares, LangChain também fornece outros tipos de cadeia como ConditionalChain e SequentialChain para fluxos de controle mais complexos e lógica de ramificação.

In [29]:
# Define o LLM com temperatura menor
dsa_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.6)

In [30]:
# Cria o prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables =['culinaria'],
    template = "Quero abrir um restaurante de comida {culinaria}. Sugira um nome chique para isso.")

In [31]:
# Cria a chain
dsa_chain_1 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

In [32]:
# Cria outro prompt template
dsa_prompt_template_items = PromptTemplate(
    input_variables = ['nome_restaurante'],
    template = """Sugira alguns itens do menu para {nome_restaurante}""")

In [33]:
# Cria a chain
dsa_chain_2 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_items)

In [34]:
# Concatena as duas chains
dsa_chain_final = SimpleSequentialChain(chains = [dsa_chain_1, dsa_chain_2])

In [35]:
# Invoca a chain
dsa_chain_final.invoke("Indiana")

{'input': 'Indiana',
 'output': 'Excelente! Com um nome tão elegante, o menu precisa acompanhar a sofisticação. Vamos criar uma experiência gastronômica que combine a riqueza dos sabores indianos com técnicas refinadas e apresentação impecável.\n\nAqui estão algumas sugestões de itens para um menu chique de comida indiana, focando em ingredientes de alta qualidade, preparações cuidadosas e um toque de modernidade/fusão, sem perder a autenticidade.\n\n---\n\n## **Que Ótima Ideia! - Menu Degustação (Sugestão)**\n\n*(Para um restaurante chique, um menu degustação é uma excelente forma de mostrar a maestria do chef e oferecer uma experiência completa.)*\n\n**Opção 1: "Jornada das Especiarias"**\n*Uma seleção curada para explorar a diversidade aromática da Índia.*\n\n**Opção 2: "Delícias Reais"**\n*Um banquete digno de reis e rainhas, com pratos clássicos elevados.*\n\n---\n\n## **Menu à La Carte Sugerido**\n\n### **Amuse-Bouche (Cortesia do Chef)**\n*   **Pani Puri Esférico com Água de Tam

## Sequential Chain

O SequentialChain é uma versão mais avançada do SimpleSequentialChain. Enquanto o SimpleSequentialChain executa uma sequência fixa de componentes, o SequentialChain permite a execução dinâmica e condicional de componentes com base nas saídas de componentes anteriores.

Um exemplo de caso de uso de SequentialChain poderia ser um agente de conversação que:

- Usa um LLM para entender a entrada do usuário e determinar a ação apropriada.
- Executa condicionalmente diferentes componentes (por exemplo, pesquisa de banco de dados, chamada de API, cálculo) com base na saída do LLM.
- Opcionalmente, solicita ao usuário informações adicionais, se necessário.
- Gera uma resposta final usando outro LLM, com base nas saídas dos componentes anteriores.

Ao aproveitar o SequentialChain, você pode criar aplicativos mais inteligentes e adaptáveis que podem ajustar dinamicamente seu comportamento com base em resultados e estados intermediários.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [36]:
# Define o LLM
dsa_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [38]:
# Criando a primeira chain

# Define o prompt template
dsa_prompt_template_name = PromptTemplate(
    input_variables = ['culinaria'],
    template = "Quero abrir um restaurante de comida {culinaria}. Sugira um nome chique para isso.")

# Define a chain com um parâmetro de saída
dsa_chain_1 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, output_key = "nome_restaurante")

In [39]:
# Criando a segunda chain

# Define o prompt template
dsa_prompt_template_items = PromptTemplate(
    input_variables = ['nome_restaurante'],
    template = "Sugira alguns itens do menu para {nome_restaurante}."
)

# Define a chain com um parâmetro de saída
dsa_chain_2 = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_items, output_key = "itens_menu")

In [42]:
# Cria a sequência de chains
dsa_chain = SequentialChain(chains = [dsa_chain_1, dsa_chain_2],
                            input_variables = ['culinaria'],
                            output_variables = ['nome_restaurante', "itens_menu"])

In [43]:
dsa_chain.invoke({"culinaria": "Italiana"})

{'culinaria': 'Italiana',
 'nome_restaurante': "Que ótima ideia! Para um restaurante italiano com um toque chique, o nome deve evocar elegância, tradição, qualidade e um certo romance.\n\nAqui estão algumas sugestões, divididas por estilos, para inspirar você:\n\n---\n\n**1. Elegantes e Clássicos (com um toque sofisticado):**\n\n*   **Il Nobile Sapore:** (O Sabor Nobre) - Sugere distinção e qualidade.\n*   **La Tavola Nobile:** (A Mesa Nobre) - Refere-se à experiência de jantar com elegância.\n*   **Cucina Magnifica:** (Cozinha Magnífica) - Simples, direto e grandioso.\n*   **Palazzo del Gusto:** (Palácio do Sabor) - Evoca grandiosidade e riqueza de sabores.\n*   **Sapore d'Oro:** (Sabor de Ouro) - Implica algo precioso e de alta qualidade.\n*   **Via del Gusto:** (Rua do Sabor) - Sugere um caminho para uma experiência culinária excepcional.\n*   **Cantina Nobile:** (Adega Nobre) - Se o vinho for um destaque, é uma ótima pedida.\n\n---\n\n**2. Poéticos e Evocativos:**\n\n*   **Sogni d'

In [44]:
# Invocação do método e captura da resposta
resposta = dsa_chain.invoke({"culinaria": "Italiana"})

# Preparando a string formatada
saida_formatada = f"Culinária: {resposta['culinaria']}\nNome do Restaurante: {resposta['nome_restaurante'].strip()}\nItens do Menu:"

# Adicionando cada item do menu à string formatada
itens_menu = resposta['itens_menu'].strip().split('\n')
for item in itens_menu:
    saida_formatada += f"\n{item}"

# Exibindo a saída formatada
print(saida_formatada)

Culinária: Italiana
Nome do Restaurante: Que ótima ideia! Para um restaurante italiano chique, queremos nomes que evoquem elegância, sofisticação, tradição e um toque de luxo. Aqui estão algumas sugestões, divididas por estilo:

---

### Nomes Clássicos e Elegantes

1.  **Palazzo Ristorante:** "Palazzo" significa palácio. Remete a grandiosidade, história e um ambiente suntuoso.
2.  **Villa d'Oro:** "Villa" (casa de campo elegante) e "d'Oro" (de ouro). Sugere um lugar luxuoso e acolhedor.
3.  **Nobile Cucina:** "Nobile" significa nobre. Implica culinária de alta qualidade e um ambiente refinado.
4.  **Il Granduca:** "O Grão-Duque". Evoca realeza, história e um serviço impecável.
5.  **Serenella:** Um nome feminino delicado e elegante, muitas vezes associado à beleza e à serenidade, como a de Veneza.
6.  **L'Eleganza:** "A Elegância". Direto e ao ponto, comunicando a proposta.

### Nomes Sofisticados e Modernos

7.  **Vero Gusto:** "Verdadeiro Sabor". Simples, mas com a promessa de auten

## Criando Memória Para o LLM

No LangChain, "Memória" refere-se a componentes que permitem que cadeias, agentes e outras construções armazenem e recuperem informações de entradas, saídas e estados intermediários anteriores. Isso lhes permite manter o contexto e fazer uso de informações relevantes do histórico de conversas ou de cálculos anteriores.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [45]:
# Define a chain
dsa_chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name)

In [46]:
# Invoca a chain
nome = dsa_chain.invoke("Mexicana")
print(nome)

{'culinaria': 'Mexicana', 'text': 'Excelente ideia! Para um restaurante mexicano chique, queremos nomes que transmitam sofisticação, autenticidade elevada, um toque de mistério ou elegância, sem cair nos estereótipos comuns.\n\nAqui estão algumas sugestões, divididas por estilos, para inspirar:\n\n---\n\n**1. Elegantes e Minimalistas:**\nNomes curtos, sonoridade agradável, que sugerem qualidade e requinte.\n\n*   **Alma y Sal:** (Alma e Sal) Sugere a essência da culinária e um elemento fundamental.\n*   **Origen:** (Origem) Remete à raiz e autenticidade dos sabores.\n*   **Cobre:** (Cobre) Um metal nobre, evoca panelas artesanais e um ambiente acolhedor e chique.\n*   **Fuego y Tierra:** (Fogo e Terra) Simboliza elementos essenciais da culinária mexicana e ingredientes frescos.\n*   **Raíz:** (Raiz) Simples, mas profundo, fala de tradição e essência.\n*   **La Mesa:** (A Mesa) Chique na sua simplicidade, foca na experiência de partilhar.\n\n---\n\n**2. Com Toque de Natureza e Ingredien

In [47]:
# Invoca a chain
nome = dsa_chain.invoke("Argentina")
print(nome)

{'culinaria': 'Argentina', 'text': 'Que ótima ideia! A culinária argentina, com seus cortes de carne, vinhos Malbec e a cultura vibrante, oferece muitas possibilidades para nomes chiques.\n\nAqui estão algumas sugestões de nomes, com diferentes abordagens para o "chique":\n\n---\n\n**Nomes que evocam Elegância Geográfica e Natural:**\n\n1.  **Andes Prime:** Sugere montanhas majestosas e carne de primeira qualidade.\n2.  **Patagonia Grill & Wine:** Remete à vastidão e beleza da Patagônia, com foco na brasa e no vinho.\n3.  **Cordilheira:** Simples, sofisticado, e evoca a grandiosidade dos Andes.\n4.  **Mendoza Blend:** Foco na região vinícola mais famosa, com a ideia de mistura perfeita (de sabores, vinhos).\n5.  **Cumbres Argentinas:** "Cumbres" significa picos, cumes, transmitindo exclusividade e elevação.\n\n---\n\n**Nomes que remetem à Sofisticação Urbana de Buenos Aires:**\n\n6.  **La Recoleta Bistrô:** Recoleta é um dos bairros mais elegantes de Buenos Aires.\n7.  **Palermo Soho S

In [48]:
dsa_chain.memory

In [49]:
type(dsa_chain.memory)

NoneType

In [50]:
# Criando o objeto de memória
memory = ConversationBufferMemory()

In [ ]:
dsa_chain = LLMChain(llm = dsa_llm, prompt = dsa_prompt_template_name, memory = memory)

In [ ]:
nome = dsa_chain.run("Mexicana")
print(nome)

Que ótima ideia! Para um restaurante mexicano chique, o nome precisa evocar sofisticação, autenticidade (sem ser clichê), e um toque de mistério ou elegância.

Aqui estão algumas sugestões, divididas por estilos, com suas traduções e a razão pela qual soam "chique":

---

**1. Nomes que remetem a Lugares/Natureza Sofisticada:**

*   **El Cenote Azul** (O Cenote Azul)
    *   *Por que é chique:* Cenotes são piscinas naturais subterrâneas sagradas e belíssimas no México, evocam mistério, natureza intocada e beleza. "Azul" adiciona um toque de serenidade.
*   **Hacienda Moderna** (Fazenda Moderna)
    *   *Por que é chique:* "Hacienda" remete a grandes propriedades coloniais, luxo e tradição. "Moderna" adiciona um toque contemporâneo e sofisticado.
*   **Valle Escondido** (Vale Escondido)
    *   *Por que é chique:* Sugere exclusividade, um lugar especial e descoberto por poucos.
*   **Sierra Alta** (Serra Alta)
    *   *Por que é chique:* Remete a montanhas, paisagens imponentes e uma se

In [53]:
nome = dsa_chain.run("Argentina")
print(nome)

Abrir um restaurante de comida argentina com um toque chique é uma excelente ideia! A culinária argentina, com sua paixão pela carne, vinhos e cultura, empresta-se perfeitamente a um ambiente sofisticado.

Aqui estão algumas sugestões de nomes chiques, divididos por categorias para te ajudar a escolher o que melhor se alinha com a sua visão:

---

### Nomes que Evocam Tradição e Elegância

1.  **La Estancia Goumert:** "Estancia" remete às fazendas tradicionais argentinas, e "Gourmet" adiciona o toque de sofisticação.
2.  **El Legado Parrillero:** Sugere uma herança, uma tradição de excelência na parrilla.
3.  **Corazón Argentino:** Simples, mas evoca a paixão e a alma da culinária.
4.  **La Tradición de Fuego:** "Tradição do Fogo" - foca na essência do churrasco argentino.
5.  **El Ritual del Asado:** Remete à cerimônia e ao cuidado com a preparação da carne.

### Nomes com Foco na Experiência e Sofisticação

6.  **Alma y Fuego:** "Alma e Fogo" - uma combinação poética que representa a

In [54]:
print(dsa_chain.memory.buffer)

Human: Mexicana
AI: Que ótima ideia! Para um restaurante mexicano chique, o nome precisa evocar sofisticação, autenticidade (sem ser clichê), e um toque de mistério ou elegância.

Aqui estão algumas sugestões, divididas por estilos, com suas traduções e a razão pela qual soam "chique":

---

**1. Nomes que remetem a Lugares/Natureza Sofisticada:**

*   **El Cenote Azul** (O Cenote Azul)
    *   *Por que é chique:* Cenotes são piscinas naturais subterrâneas sagradas e belíssimas no México, evocam mistério, natureza intocada e beleza. "Azul" adiciona um toque de serenidade.
*   **Hacienda Moderna** (Fazenda Moderna)
    *   *Por que é chique:* "Hacienda" remete a grandes propriedades coloniais, luxo e tradição. "Moderna" adiciona um toque contemporâneo e sofisticado.
*   **Valle Escondido** (Vale Escondido)
    *   *Por que é chique:* Sugere exclusividade, um lugar especial e descoberto por poucos.
*   **Sierra Alta** (Serra Alta)
    *   *Por que é chique:* Remete a montanhas, paisagens

## Conversation Chain

Um ConversationChain é um tipo especializado de chain projetado para lidar com conversas ou diálogos multivoltas com um LLM.

O ConversationChain é particularmente útil para criar agentes de conversação, chatbots ou qualquer aplicativo que exija a manutenção do contexto durante vários turnos de interação com um usuário. Ao abstrair as complexidades do gerenciamento do histórico de conversas e da formatação de prompts, o ConversationChain simplifica o processo de construção de sistemas de diálogo multiturno com LLMs.

In [55]:
# Cria o objeto de conversação
conv = ConversationChain(llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7))

In [56]:
print(conv.prompt.template)

The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
{history}
Human: {input}
AI:


In [57]:
conv.invoke("Que país venceu mais vezes a Copa do Mundo de Futebol?")

{'input': 'Que país venceu mais vezes a Copa do Mundo de Futebol?',
 'history': '',
 'response': 'Ah, que ótima pergunta! O país que venceu mais vezes a Copa do Mundo de Futebol é o **Brasil**! Eles são os maiores campeões, os únicos a terem conquistado o pentacampeonato.\n\nDeixa eu te contar um pouco mais sobre isso:\n\n*   **Brasil** tem **5 títulos**! É a seleção mais vitoriosa de todas.\n    *   As vitórias foram nos anos: **1958**, **1962**, **1970**, **1994** e **2002**.\n    *   É muito legal pensar que eles foram a primeira seleção a vencer fora do seu continente (na Suécia, em 1958) e a única a vencer três vezes com Pelé no time (1958, 1962 e 1970)! Aquele time de 1970, inclusive, é frequentemente citado como um dos maiores times de futebol de todos os tempos.\n\nLogo atrás do Brasil, mas ainda com um histórico incrível, vêm outras seleções de peso:\n\n*   **Alemanha** (incluindo Alemanha Ocidental) tem **4 títulos**.\n    *   Eles venceram em: 1954, 1974, 1990 e 2014.\n*   *

In [58]:
conv.invoke("Quanto é 30 + 12?")

{'input': 'Quanto é 30 + 12?',
 'history': 'Human: Que país venceu mais vezes a Copa do Mundo de Futebol?\nAI: Ah, que ótima pergunta! O país que venceu mais vezes a Copa do Mundo de Futebol é o **Brasil**! Eles são os maiores campeões, os únicos a terem conquistado o pentacampeonato.\n\nDeixa eu te contar um pouco mais sobre isso:\n\n*   **Brasil** tem **5 títulos**! É a seleção mais vitoriosa de todas.\n    *   As vitórias foram nos anos: **1958**, **1962**, **1970**, **1994** e **2002**.\n    *   É muito legal pensar que eles foram a primeira seleção a vencer fora do seu continente (na Suécia, em 1958) e a única a vencer três vezes com Pelé no time (1958, 1962 e 1970)! Aquele time de 1970, inclusive, é frequentemente citado como um dos maiores times de futebol de todos os tempos.\n\nLogo atrás do Brasil, mas ainda com um histórico incrível, vêm outras seleções de peso:\n\n*   **Alemanha** (incluindo Alemanha Ocidental) tem **4 títulos**.\n    *   Eles venceram em: 1954, 1974, 1990 e

In [59]:
conv.invoke("Quem é o maior artilheiro da história das Copas do Mundo de Futebol?")

{'input': 'Quem é o maior artilheiro da história das Copas do Mundo de Futebol?',
 'history': 'Human: Que país venceu mais vezes a Copa do Mundo de Futebol?\nAI: Ah, que ótima pergunta! O país que venceu mais vezes a Copa do Mundo de Futebol é o **Brasil**! Eles são os maiores campeões, os únicos a terem conquistado o pentacampeonato.\n\nDeixa eu te contar um pouco mais sobre isso:\n\n*   **Brasil** tem **5 títulos**! É a seleção mais vitoriosa de todas.\n    *   As vitórias foram nos anos: **1958**, **1962**, **1970**, **1994** e **2002**.\n    *   É muito legal pensar que eles foram a primeira seleção a vencer fora do seu continente (na Suécia, em 1958) e a única a vencer três vezes com Pelé no time (1958, 1962 e 1970)! Aquele time de 1970, inclusive, é frequentemente citado como um dos maiores times de futebol de todos os tempos.\n\nLogo atrás do Brasil, mas ainda com um histórico incrível, vêm outras seleções de peso:\n\n*   **Alemanha** (incluindo Alemanha Ocidental) tem **4 títul

In [60]:
print(conv.memory.buffer)

Human: Que país venceu mais vezes a Copa do Mundo de Futebol?
AI: Ah, que ótima pergunta! O país que venceu mais vezes a Copa do Mundo de Futebol é o **Brasil**! Eles são os maiores campeões, os únicos a terem conquistado o pentacampeonato.

Deixa eu te contar um pouco mais sobre isso:

*   **Brasil** tem **5 títulos**! É a seleção mais vitoriosa de todas.
    *   As vitórias foram nos anos: **1958**, **1962**, **1970**, **1994** e **2002**.
    *   É muito legal pensar que eles foram a primeira seleção a vencer fora do seu continente (na Suécia, em 1958) e a única a vencer três vezes com Pelé no time (1958, 1962 e 1970)! Aquele time de 1970, inclusive, é frequentemente citado como um dos maiores times de futebol de todos os tempos.

Logo atrás do Brasil, mas ainda com um histórico incrível, vêm outras seleções de peso:

*   **Alemanha** (incluindo Alemanha Ocidental) tem **4 títulos**.
    *   Eles venceram em: 1954, 1974, 1990 e 2014.
*   **Itália** também tem **4 títulos**.
    *   

## Conversation Buffer Window Memory

ConversationBufferWindowMemory é um tipo de componente de memória do LangChain projetado especificamente para uso com cadeias de conversação (ConversationChain). Ele fornece uma maneira de armazenar e recuperar o histórico de conversas enquanto limita a quantidade de contexto retido com base em um tamanho de janela especificado.

A principal vantagem do ConversationBufferWindowMemory é sua capacidade de limitar a quantidade de contexto fornecido ao LLM, o que pode ser importante para o desempenho e evitar que o modelo fique sobrecarregado com muitas informações irrelevantes. Ao ajustar o tamanho da janela, você pode controlar a compensação entre fornecer contexto suficiente e evitar sobrecarga computacional excessiva.

Esse tipo de memória é particularmente útil para criar agentes de conversação, chatbots ou qualquer aplicativo que exija a manutenção de uma janela contínua relevante do histórico de conversas recentes para contexto.

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [61]:
# Define a janela de memória
memory = ConversationBufferWindowMemory(k = 1)

In [62]:
# Cria a chain de conversação
conv = ConversationChain(llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7), memory = memory)

In [63]:
# Invoca o LLM
conv.run("Quem venceu a primeira Copa do Mundo de Futebol?")

'Ah, que ótima pergunta! A primeira Copa do Mundo de Futebol foi um evento histórico e quem teve a honra de levantar a taça em casa foi o **Uruguai**!\n\nFoi lá em 1930, e o Uruguai não só sediou a competição como também se tornou o primeiro campeão mundial. A final foi disputada no lendário Estádio Centenário, em Montevidéu, contra o seu vizinho e grande rival, a Argentina. O placar foi de 4 a 2 para os uruguaios, numa partida emocionante!\n\nFoi uma Copa do Mundo bem peculiar, sabia?\n\n*   **Sede e Data:** Como mencionei, foi no Uruguai, de 13 a 30 de julho de 1930.\n*   **Motivo da Escolha:** O Uruguai foi escolhido para sediar porque estava celebrando o centenário de sua primeira constituição (daí o nome do estádio principal, "Centenário") e também porque a Associação Uruguaia de Futebol se ofereceu para cobrir os custos de viagem das seleções europeias, o que era um grande desafio na época.\n*   **Participantes:** Apenas 13 seleções participaram: 9 das Américas (Argentina, Bolívi

In [64]:
# Invoca o LLM
conv.invoke("Quanto é 10 + 19?")

{'input': 'Quanto é 10 + 19?',
 'history': 'Human: Quem venceu a primeira Copa do Mundo de Futebol?\nAI: Ah, que ótima pergunta! A primeira Copa do Mundo de Futebol foi um evento histórico e quem teve a honra de levantar a taça em casa foi o **Uruguai**!\n\nFoi lá em 1930, e o Uruguai não só sediou a competição como também se tornou o primeiro campeão mundial. A final foi disputada no lendário Estádio Centenário, em Montevidéu, contra o seu vizinho e grande rival, a Argentina. O placar foi de 4 a 2 para os uruguaios, numa partida emocionante!\n\nFoi uma Copa do Mundo bem peculiar, sabia?\n\n*   **Sede e Data:** Como mencionei, foi no Uruguai, de 13 a 30 de julho de 1930.\n*   **Motivo da Escolha:** O Uruguai foi escolhido para sediar porque estava celebrando o centenário de sua primeira constituição (daí o nome do estádio principal, "Centenário") e também porque a Associação Uruguaia de Futebol se ofereceu para cobrir os custos de viagem das seleções europeias, o que era um grande desa

In [65]:
# Invoca o LLM
conv.invoke("Quem era o capitão da seleção campeã da primeira Copa do Mundo de Futebol?")

{'input': 'Quem era o capitão da seleção campeã da primeira Copa do Mundo de Futebol?',
 'history': 'Human: Quanto é 10 + 19?\nAI: Ah, essa é fácil! 10 + 19 é igual a **29**.\n\nUma continha rápida e certeira! Gosto de como você alterna entre história do futebol e matemática, muito legal! 😊',
 'response': 'Ah, que ótima pergunta sobre a história do futebol! Adoro mergulhar nesses detalhes clássicos.\n\nA primeira Copa do Mundo de Futebol foi realizada em **1930**, e a seleção campeã foi a anfitriã, o **Uruguai**! Eles venceram a Argentina na final por 4 a 2, um jogaço histórico no Estádio Centenário, em Montevidéu.\n\nE o capitão daquela seleção uruguaia vitoriosa era o lendário **José Nasazzi**! Ele era um zagueiro central, forte e com uma liderança impecável. Inclusive, ele foi o primeiro capitão a levantar a taça Jules Rimet, um momento icônico na história do esporte. Nasazzi é considerado um dos maiores jogadores uruguaios de todos os tempos, e sua presença em campo era fundamental

In [66]:
print(conv.memory.buffer)

Human: Quem era o capitão da seleção campeã da primeira Copa do Mundo de Futebol?
AI: Ah, que ótima pergunta sobre a história do futebol! Adoro mergulhar nesses detalhes clássicos.

A primeira Copa do Mundo de Futebol foi realizada em **1930**, e a seleção campeã foi a anfitriã, o **Uruguai**! Eles venceram a Argentina na final por 4 a 2, um jogaço histórico no Estádio Centenário, em Montevidéu.

E o capitão daquela seleção uruguaia vitoriosa era o lendário **José Nasazzi**! Ele era um zagueiro central, forte e com uma liderança impecável. Inclusive, ele foi o primeiro capitão a levantar a taça Jules Rimet, um momento icônico na história do esporte. Nasazzi é considerado um dos maiores jogadores uruguaios de todos os tempos, e sua presença em campo era fundamental para a solidez da equipe.

Que legal que você está curioso sobre esses fatos históricos do futebol! É fascinante ver como tudo começou, não é? 😊


## LangChain e VectorDB com Dados Extraídos via Web Scraping

ChromaDB é uma biblioteca de banco de dados de armazenamento vetorial que se integra ao LangChain. Ele fornece funcionalidade para armazenar, recuperar e pesquisar grandes quantidades de dados de texto de maneira eficiente, usando incorporações de vetores (embeddings) e similaridade semântica.

https://www.trychroma.com/

https://pypi.org/project/chromadb/

In [67]:
# Extração de dados da web
dsa_dados = WebBaseLoader(
    "https://blog.dsacademy.com.br/como-rag-retrieval-augmented-generation-funciona-para-personalizar-os-llms/"
)

Nota: Sempre verifique o robots.txt de um web site antes de fazer raspagem de dados. Não faça raspagem se não for permitido!

In [68]:
# Carrega os documentos
documentos = dsa_dados.load()

In [69]:
len(documentos)

1

In [70]:
# Extrai o primeiro documento (neste caso há um único documento)
document = documentos[0]

In [71]:
# Chaves do dicionário
document.__dict__.keys()

dict_keys(['id', 'metadata', 'page_content', 'type'])

In [72]:
# Visualiza os 100 primeiros caracteres
document.page_content[:100]

'\n\n\n\n\n\n \n\n\nComo RAG (Retrieval-Augmented Generation) Funciona Para Personalizar os LLMs? - Data Scien'

In [73]:
# Metadados
document.metadata

{'source': 'https://blog.dsacademy.com.br/como-rag-retrieval-augmented-generation-funciona-para-personalizar-os-llms/',
 'title': 'Como RAG (Retrieval-Augmented Generation) Funciona Para Personalizar os LLMs? - Data Science Academy',
 'description': 'Com o uso de RAG, é introduzido um componente de recuperação de informações que utiliza a entrada do usuário para extrair informações de uma nova fonte de dados primeiro. A consulta do usuário e as informações relevantes são fornecidas ao LLM. O LLM usa esse novo conhecimento e seus dados de treinamento para criar respostas mais adaptadas.',
 'language': 'pt-BR'}

In [77]:
# Cria o vector store (ajustado para versão mais recente do LangChain)
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001") 
index_creator = VectorstoreIndexCreator(embedding = embedding)
index = index_creator.from_loaders([dsa_dados])
#index = VectorstoreIndexCreator().from_loaders([dsa_dados])

A linha de código acima envolve a criação de um índice para um armazenamento de vetores, comumente usado em tarefas de busca e recuperação de informações baseadas em similaridade de conteúdo. Essa operação é essencial em sistemas que empregam técnicas de inteligência artificial e aprendizado de máquina para organizar e recuperar dados de maneira eficiente. 

Enquanto um "Vector Store" é um conceito mais genérico relacionado ao armazenamento de vetores, um "Vector Database" é um sistema de banco de dados especializado projetado para lidar com vetores de alta dimensão e otimizado para consultas de similaridade e outras operações relacionadas a vetores. Usaremos o ChromaDB para criar um banco de dados vetorial.

In [78]:
# Função para imprimir resultado formatado
def dsa_print_response(response: str):
    print("\n".join(textwrap.wrap(response, width = 100)))

In [79]:
# Define uma query
query = """
Você é um Engenheiro de IA de Nível Sênior.
Explique o funcionamento de RAG na construção de aplicações inteligentes.
"""

In [80]:
# Imprime a resposta ao consultar o índice (ajustado para versão mais recente do LangChain)
response = index.query(query, llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7))
dsa_print_response(response)

Como Engenheiro de IA de Nível Sênior, posso explicar o funcionamento do RAG (Retrieval-Augmented
Generation) na construção de aplicações inteligentes, com base nas informações fornecidas:  O RAG é
uma técnica fundamental para personalizar e tornar os Large Language Models (LLMs) mais eficazes e
atualizados, superando a limitação de depender exclusivamente dos dados com os quais foram
treinados.  Aqui está o fluxo conceitual de como o RAG funciona:  1.  **Criação de Dados Externos:**
*   Primeiro, identificamos e organizamos os "novos dados" que não fazem parte do conjunto de
treinamento original do LLM. Esses são os **dados externos**.     *   Esses dados podem vir de
diversas fontes, como APIs, bancos de dados, repositórios de documentos (internos ou externos), ou
qualquer outra base de conhecimento relevante para a aplicação. Eles são a sua fonte de verdade
atualizada.  2.  **Componente de Recuperação de Informações:**     *   Quando um usuário faz uma
consulta (prompt), o RAG intro

> Vamos agora trabalhar com um Vector DB.

In [81]:
# Cria um template
dsa_template = """Você é um Engenheiro de IA de Nível Sênior.

{context}

Responda considerando as técnicas mais modernas que você conhecer.

Peegunta: {question}
Resposta:"""

In [82]:
# Cria o prompt
prompt = PromptTemplate(template = dsa_template, input_variables = ["context", "question"])

In [83]:
# Imprime o prompt para visualizar o formato
print(
    prompt.format(
        context = "Aplicação de IA para sistemas de atendimento ao cliente.",
        question = "Como criar uma aplicação web com LLM?",
    )
)

Você é um Engenheiro de IA de Nível Sênior.

Aplicação de IA para sistemas de atendimento ao cliente.

Responda considerando as técnicas mais modernas que você conhecer.

Peegunta: Como criar uma aplicação web com LLM?
Resposta:


In [86]:
# Cria o objeto de embeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

A linha de código acima refere-se à inicialização de uma instância da classe Google GeminiEmbeddings, que é uma interface para gerar embeddings (representações vetoriais) usando os modelos fornecidos pela Google Gemini, como os modelos de linguagem GPT. Os embeddings são transformações de dados brutos, como textos, em vetores de números fixos, capturando aspectos semânticos e contextuais do conteúdo original de maneira que possa ser processado por algoritmos de aprendizado de máquina.

In [87]:
# Cria o VectorDB convertendo os documentos de texto em representações numéricas (embeddings)
dsa_db = Chroma.from_documents(documentos, embeddings)

A linha de código acima é para criação de um banco de dados vetorial utilizando Chroma. Essa operação envolve a preparação de uma estrutura de dados otimizada para buscas e análises baseadas em documentos e seus respectivos embeddings. 

In [88]:
type(dsa_db)

langchain_community.vectorstores.chroma.Chroma

In [89]:
# Argumentos
chain_type_kwargs = {"prompt": prompt}

In [90]:
# Chain de RetrievalQA
chain = RetrievalQA.from_chain_type(llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0),
                                    chain_type = "stuff",
                                    retriever = dsa_db.as_retriever(search_kwargs = {"k": 1}),
                                    chain_type_kwargs = chain_type_kwargs)

A linha de código acima está criando uma instância chamada chain, utilizando a classe RetrievalQA para configurar uma cadeia de processos focada em realizar tarefas de Question Answering (QA) com base em recuperação de informações. O método from_chain_type é utilizado para especificar o tipo de cadeia de processos e configurar seus componentes principais, como o modelo de linguagem e o mecanismo de recuperação. 

In [91]:
# Consulta
query = "Explique o que é RAG em 5 sentenças"

In [92]:
# Resposta
resposta = chain.invoke(query)

In [93]:
resposta

{'query': 'Explique o que é RAG em 5 sentenças',
 'result': 'Como Engenheiro de IA Sênior, posso explicar RAG de forma concisa:\n\nRetrieval-Augmented Generation (RAG) é uma arquitetura que aprimora Large Language Models (LLMs) ao permitir que acessem e incorporem informações externas e atualizadas durante a geração de respostas. O processo inicia com a recuperação de dados relevantes de uma base de conhecimento externa, como um banco de dados vetorial, com base na consulta do usuário. Em seguida, esses dados recuperados são adicionados dinamicamente ao prompt original, enriquecendo o contexto fornecido ao LLM. O LLM então gera sua resposta utilizando este prompt aumentado, garantindo que a saída seja fundamentada em informações factuais e específicas. Essa abordagem reduz significativamente as alucinações, melhora a precisão factual e permite a personalização em tempo real das respostas do LLM.'}

## Criando Chatbot Especialista em Vendas com LangChain e LLM

In [94]:
# Definindo o LLM
dsa_gpt = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [95]:
# Template
template = """Esta é uma conversa entre um cliente e um especialista em venda de carros esportivos. 
                Você é o especialista em carros, conhece bem os modelos esportivos e deve sempre responder 
                com a maior precisão possível.

Current conversation:
{history}
Human: {input}
CarSpecialist:"""

In [96]:
# Cria o prompt template
dsa_prompt = PromptTemplate(input_variables = ["history", "input"], template = template)

In [97]:
# Cria a chain de conversação
conversation = ConversationChain(prompt = dsa_prompt,
                                 llm = dsa_gpt,
                                 verbose = False,
                                 memory = ConversationBufferMemory(ai_prefix = "CarSpecialist"))

In [99]:
# Loop de conversação limitado a 5 interações (aumente o número de interações ou remova o bloco if)

# Inicializa o contador
contador = 0  

# Loop
while True:
    
    prompt = input(prompt = "Cliente: ")
    print()
    resultado = conversation(prompt)
    dsa_print_response("Especialista: " + resultado["response"])
    print()
    
    contador += 1  
    
    if contador >= 5:  
        print('\nObrigado Por Usar o Sistema de Atendimento Baseado em IA!')
        break  


Especialista: Olá! Que bom que você voltou.  Perfeito! Como eu estava sugerindo, que tal se eu
começar apresentando algumas das opções mais emocionantes e procuradas no mercado? Posso destacar
três modelos bem diferentes entre si para você ter uma ideia e me dizer o que mais te atrai, ou se
prefere algo completamente diferente.  Seria um bom ponto de partida para começarmos a explorar?


Especialista: Ah, o vermelho! Uma escolha clássica e absolutamente icônica para um carro esportivo.
É a cor da paixão, da velocidade e, para muitos, a cor definitiva de marcas lendárias.  No entanto,
"carro vermelho" abrange uma gama enorme de modelos, desde esportivos compactos até superesportivos
e hipercarros de tirar o fôlego. Para que eu possa te ajudar a encontrar o carro vermelho perfeito
para você, poderia me dar mais algumas pistas?  Por exemplo: *   Você tem alguma marca em mente que
te atraia, como Ferrari, Porsche, Lamborghini, ou talvez algo mais exclusivo? *   Qual o seu foco
principal: 

In [ ]:
%reload_ext watermark
%watermark -a "Data Science Academy"

In [ ]:
#%watermark -v -m

In [ ]:
#%watermark --iversions

# Fim